<a href="https://colab.research.google.com/github/galamahb/Final-Assignment/blob/main/Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# ---------------------------------------------------------
# Code Cell 1
# Imports, custom errors, and user account classes.
# This part sets up the libraries and the account types used in the system.
# ---------------------------------------------------------

import os  # Used to create folders and file paths for saved pickle data.
import pickle  # Used to save and load users, facilities, and bookings.
import tkinter as tk  # Used to build the GUI window, labels, buttons, and entries.
from tkinter import ttk, messagebox  # ttk is used for tables/dropdowns, messagebox is used for pop-up messages.


class BookingError(Exception):
    # This error is used when something goes wrong with booking rules.
    # Example: a Standard student trying to book a Premium facility.
    pass


class LoginError(Exception):
    # This error is used when login fails.
    # Example: wrong user ID or password.
    pass


class ValidationError(Exception):
    # This error is used when the user enters invalid input.
    # Example: empty name, invalid email, or wrong access type.
    pass


class User:
    # This is the parent class for all accounts.
    # Student and Administrator both share these details.

    def __init__(self, user_id, name, email, password):
        self.__user_id = user_id  # Stores the unique ID used for login and records.
        self.__name = name  # Stores the user's name for dashboards and receipts.
        self.__email = email  # Stores the user's email for account details.
        self.__password = password  # Stores the password used for login checking.

    def get_user_id(self):
        return self.__user_id  # Returns the user's ID.

    def get_name(self):
        return self.__name  # Returns the user's name.

    def set_name(self, name):
        if name.strip() == "":  # Checks if the name is empty or only spaces.
            raise ValidationError("Name cannot be empty.")  # Stops empty names from being saved.

        self.__name = name  # Saves the new valid name.

    def get_email(self):
        return self.__email  # Returns the user's email.

    def set_email(self, email):
        if "@" not in email:  # Simple email validation.
            raise ValidationError("Email must contain @.")  # Stops invalid email from being saved.

        self.__email = email  # Saves the new valid email.

    def check_password(self, password):
        return self.__password == password  # Returns True if typed password matches saved password.

    def get_role(self):
        return "User"  # Student and Administrator override this method.

    def get_access_type(self):
        return "Not Applicable"  # Admins do not have Standard/Premium access.


class Student(User):
    # Student inherits from User.
    # It adds access_type because students can be Standard or Premium.

    def __init__(self, user_id, name, email, password, access_type):
        super().__init__(user_id, name, email, password)  # Sends common user details to User class.
        self.__access_type = access_type  # Stores Standard or Premium.

    def get_access_type(self):
        return self.__access_type  # Returns the student's access type.

    def set_access_type(self, access_type):
        if access_type != "Standard" and access_type != "Premium":  # Only two access types are allowed.
            raise ValidationError("Access type must be Standard or Premium.")

        self.__access_type = access_type  # Saves the valid access type.

    def get_role(self):
        return "Student"  # This tells the GUI to open the student dashboard.


class Administrator(User):
    # Administrator also inherits from User.
    # Admins manage users, bookings, facilities, and reports.

    def __init__(self, user_id, name, email, password):
        super().__init__(user_id, name, email, password)  # Admin still has the same basic account details.

    def get_role(self):
        return "Administrator"  # This tells the GUI to open the admin dashboard.


In [ ]:
# ---------------------------------------------------------
# Code Cell 2
# Facility, Booking, and Storage classes.
# This part stores facility details, booking records, and pickle files.
# ---------------------------------------------------------

class Facility:
    # This class represents one campus facility.
    # Examples: Study Room A, Basketball Court, Event Hall.

    def __init__(self, facility_id, name, facility_type, location, capacity, price_per_hour, time_slots):
        self.__facility_id = facility_id  # Stores facility ID like F1 or F2.
        self.__name = name  # Stores facility name shown to the user.
        self.__facility_type = facility_type  # Stores category like Study Room or Sports Court.
        self.__location = location  # Stores where the facility is located.
        self.__capacity = capacity  # Stores the maximum number of people allowed.
        self.__price_per_hour = price_per_hour  # Stores the hourly fee.
        self.__time_slots = time_slots  # Stores available time slots.
        self.__status = "Available"  # Facility starts as Available.

    def get_facility_id(self):
        return self.__facility_id  # Returns facility ID.

    def get_name(self):
        return self.__name  # Returns facility name.

    def get_facility_type(self):
        return self.__facility_type  # Returns the type of facility.

    def get_location(self):
        return self.__location  # Returns location.

    def get_capacity(self):
        return self.__capacity  # Returns max capacity.

    def get_price_per_hour(self):
        return self.__price_per_hour  # Returns hourly price.

    def get_time_slots(self):
        return self.__time_slots  # Returns available time slots.

    def get_status(self):
        return self.__status  # Returns Available or Unavailable.

    def set_status(self, status):
        if status != "Available" and status != "Unavailable":  # Only these two statuses are allowed.
            raise ValidationError("Status must be Available or Unavailable.")

        self.__status = status  # Saves new status.

    def calculate_cost(self, hours):
        return self.__price_per_hour * hours  # Total cost = price per hour times duration.

    def get_required_access(self):
        if self.__facility_type == "Study Room":  # Study rooms are Standard.
            return "Standard"

        return "Premium"  # Sports courts and event halls are Premium.


class Booking:
    # This class represents one booking record.
    # It connects a student, a facility, a date, a time slot, people, and cost.

    def __init__(self, booking_id, student_id, facility_id, date, time_slot, hours, number_of_people, total_cost):
        self.__booking_id = booking_id  # Stores unique booking ID.
        self.__student_id = student_id  # Stores which student made the booking.
        self.__facility_id = facility_id  # Stores which facility was booked.
        self.__date = date  # Stores selected date.
        self.__time_slot = time_slot  # Stores selected time slot.
        self.__hours = hours  # Stores duration calculated from time slot.
        self.__number_of_people = number_of_people  # Stores how many people are attending.
        self.__total_cost = total_cost  # Stores final calculated cost.
        self.__status = "Confirmed"  # New bookings start as Confirmed.

        if total_cost == 0:  # Free facility booking.
            self.__payment_status = "Free"
        else:  # Paid facility booking.
            self.__payment_status = "Paid"

    def get_booking_id(self):
        return self.__booking_id  # Returns booking ID.

    def get_student_id(self):
        return self.__student_id  # Returns student ID.

    def get_facility_id(self):
        return self.__facility_id  # Returns facility ID.

    def get_date(self):
        return self.__date  # Returns booking date.

    def get_time_slot(self):
        return self.__time_slot  # Returns time slot.

    def get_hours(self):
        return self.__hours  # Returns duration.

    def get_number_of_people(self):
        return self.__number_of_people  # Returns number of people.

    def get_total_cost(self):
        return self.__total_cost  # Returns total cost.

    def get_status(self):
        return self.__status  # Returns Confirmed or Cancelled.

    def get_payment_status(self):
        return self.__payment_status  # Returns Free or Paid.

    def cancel_booking(self):
        self.__status = "Cancelled"  # Keeps record but changes status.

    def update_booking(self, date, time_slot, hours, number_of_people, total_cost):
        self.__date = date  # Saves updated date.
        self.__time_slot = time_slot  # Saves updated time slot.
        self.__hours = hours  # Saves updated duration.
        self.__number_of_people = number_of_people  # Saves updated people count.
        self.__total_cost = total_cost  # Saves updated cost.

        if total_cost == 0:  # Updates payment status if free.
            self.__payment_status = "Free"
        else:  # Updates payment status if paid.
            self.__payment_status = "Paid"


class Storage:
    # This class handles saving and loading data using Pickle.
    # Users, facilities, and bookings are saved in separate binary files.

    def __init__(self):
        self.__folder = "smart_campus_final_data"  # Folder where pickle files are saved.
        self.__users_file = os.path.join(self.__folder, "users.pkl")  # Path for users file.
        self.__facilities_file = os.path.join(self.__folder, "facilities.pkl")  # Path for facilities file.
        self.__bookings_file = os.path.join(self.__folder, "bookings.pkl")  # Path for bookings file.

        if not os.path.exists(self.__folder):  # Checks if folder does not exist.
            os.makedirs(self.__folder)  # Creates the folder.

    def save_data(self, users, facilities, bookings):
        with open(self.__users_file, "wb") as file:  # Opens users file in write-binary mode.
            pickle.dump(users, file)  # Saves users dictionary.

        with open(self.__facilities_file, "wb") as file:  # Opens facilities file in write-binary mode.
            pickle.dump(facilities, file)  # Saves facilities dictionary.

        with open(self.__bookings_file, "wb") as file:  # Opens bookings file in write-binary mode.
            pickle.dump(bookings, file)  # Saves bookings list.

    def load_file(self, file_name, default_value):
        if os.path.exists(file_name):  # Checks if file exists.
            with open(file_name, "rb") as file:  # Opens file in read-binary mode.
                return pickle.load(file)  # Loads and returns saved data.

        return default_value  # Returns empty data if file does not exist.

    def load_data(self):
        users = self.load_file(self.__users_file, {})  # Loads users or empty dictionary.
        facilities = self.load_file(self.__facilities_file, {})  # Loads facilities or empty dictionary.
        bookings = self.load_file(self.__bookings_file, [])  # Loads bookings or empty list.

        return users, facilities, bookings  # Sends all data back to BookingSystem.

In [ ]:
# ---------------------------------------------------------
# Code Cell 3
# BookingSystem class.
# This is the main logic of the system.
# It checks login, access rules, availability, capacity, cost, and reports.
# ---------------------------------------------------------

class BookingSystem:
    def __init__(self):
        self.__storage = Storage()  # Creates storage object for pickle files.
        self.__users, self.__facilities, self.__bookings = self.__storage.load_data()  # Loads saved data.
        self.__current_user = None  # No user is logged in at the start.
        self.create_default_data()  # Adds starter data if the system is empty.

    def create_default_data(self):
        if len(self.__users) == 0:  # Adds starter users only if no users exist.
            self.__users["admin"] = Administrator("admin", "System Admin", "admin@zu.ac.ae", "admin123")
            self.__users["S100"] = Student("S100", "Noura Student", "noura@zu.ac.ae", "1234", "Standard")
            self.__users["S200"] = Student("S200", "Ahmed Student", "ahmed@zu.ac.ae", "1234", "Premium")

        if len(self.__facilities) == 0:  # Adds starter facilities only if no facilities exist.
            self.__facilities["F1"] = Facility("F1", "Study Room A", "Study Room", "Building C, Floor 2", 6, 0, ["9:00-11:00", "11:00-1:00", "2:00-4:00"])
            self.__facilities["F2"] = Facility("F2", "Study Room B", "Study Room", "Library Wing", 4, 0, ["10:00-12:00", "1:00-3:00"])
            self.__facilities["F3"] = Facility("F3", "Basketball Court", "Sports Court", "Outdoor Area", 10, 20, ["8:00-10:00", "4:00-6:00"])
            self.__facilities["F4"] = Facility("F4", "Tennis Court", "Sports Court", "Outdoor Area", 4, 15, ["9:00-11:00", "5:00-7:00"])
            self.__facilities["F5"] = Facility("F5", "Event Hall", "Event Hall", "Main Building", 50, 100, ["10:00-12:00", "2:00-4:00", "4:00-6:00"])

        self.save_all_data()  # Saves starter data.

    def save_all_data(self):
        self.__storage.save_data(self.__users, self.__facilities, self.__bookings)  # Saves all data to pickle.

    def calculate_hours_from_time_slot(self, time_slot):
        parts = time_slot.split("-")  # Splits time slot into start and end.
        start_part = parts[0]  # Stores start time.
        end_part = parts[1]  # Stores end time.
        start_hour = int(start_part.split(":")[0])  # Gets start hour number.
        end_hour = int(end_part.split(":")[0])  # Gets end hour number.

        if end_hour <= start_hour:  # Handles time like 11:00-1:00.
            end_hour = end_hour + 12  # Treats end time as afternoon.

        return end_hour - start_hour  # Returns duration.

    def register_user(self, user_id, name, email, password, user_type, access_type):
        if user_id.strip() == "" or name.strip() == "" or email.strip() == "" or password.strip() == "":
            raise ValidationError("Please fill all fields.")  # Stops empty form.

        if user_id in self.__users:
            raise ValidationError("This user ID already exists.")  # Stops duplicate ID.

        if "@" not in email:
            raise ValidationError("Please enter a valid email.")  # Stops invalid email.

        if user_type == "Student":
            self.__users[user_id] = Student(user_id, name, email, password, access_type)  # Creates student.
        elif user_type == "Administrator":
            self.__users[user_id] = Administrator(user_id, name, email, password)  # Creates admin.
        else:
            raise ValidationError("Please choose a valid user type.")  # Stops wrong user type.

        self.save_all_data()  # Saves new account.

    def login(self, user_id, password):
        if user_id not in self.__users:
            raise LoginError("User ID was not found.")  # Stops missing user ID.

        if self.__users[user_id].check_password(password) == False:
            raise LoginError("Password is incorrect.")  # Stops wrong password.

        self.__current_user = self.__users[user_id]  # Saves logged-in user.
        return self.__current_user  # Sends user back to GUI.

    def get_current_user(self):
        return self.__current_user  # Returns logged-in user.

    def get_all_users(self):
        return self.__users  # Returns all users.

    def get_all_facilities(self):
        return self.__facilities  # Returns all facilities.

    def get_all_bookings(self):
        return self.__bookings  # Returns all bookings.

    def get_facility(self, facility_id):
        return self.__facilities[facility_id]  # Returns one facility by ID.

    def update_current_user_profile(self, name, email):
        if self.__current_user is None:
            raise ValidationError("No user is logged in.")  # Stops update with no user.

        self.__current_user.set_name(name)  # Updates name.
        self.__current_user.set_email(email)  # Updates email.
        self.save_all_data()  # Saves profile changes.

    def upgrade_current_user_access(self):
        if self.__current_user is None:
            raise ValidationError("No user is logged in.")  # Stops upgrade with no user.

        if self.__current_user.get_role() != "Student":
            raise ValidationError("Only students can upgrade access.")  # Only students upgrade.

        if self.__current_user.get_access_type() == "Premium":
            raise ValidationError("You already have Premium access.")  # Stops repeated upgrade.

        self.__current_user.set_access_type("Premium")  # Changes access to Premium.
        self.save_all_data()  # Saves upgrade.

    def delete_user(self, user_id):
        user_id = str(user_id)  # Converts selected ID to string.

        if user_id not in self.__users:
            raise ValidationError("User was not found.")  # Stops missing user.

        if user_id == "admin":
            raise ValidationError("The main admin account cannot be deleted.")  # Protects main admin.

        del self.__users[user_id]  # Deletes user.
        self.save_all_data()  # Saves updated users.

    def upgrade_access(self, user_id):
        user_id = str(user_id)  # Converts selected ID to string.

        if user_id not in self.__users:
            raise ValidationError("User was not found.")  # Stops missing user.

        user = self.__users[user_id]  # Gets selected user.

        if user.get_role() != "Student":
            raise ValidationError("Only student accounts can be upgraded.")  # Stops admin upgrade.

        if user.get_access_type() == "Premium":
            raise ValidationError("This student already has Premium access.")  # Stops repeated upgrade.

        user.set_access_type("Premium")  # Upgrades student.
        self.save_all_data()  # Saves upgrade.

    def check_access_permission(self, student, facility):
        if student.get_access_type() == "Standard" and facility.get_required_access() == "Premium":
            raise BookingError("This facility requires Premium access. Please upgrade to book it.")  # Blocks Standard from Premium.

    def is_time_slot_taken(self, facility_id, date, time_slot, ignored_booking_id=""):
        for booking in self.__bookings:  # Checks every booking.
            same_booking = booking.get_booking_id() == ignored_booking_id  # Ignores same booking during modify.
            same_facility = booking.get_facility_id() == facility_id  # Checks same facility.
            same_date = booking.get_date() == date  # Checks same date.
            same_time = booking.get_time_slot() == time_slot  # Checks same time.
            active_booking = booking.get_status() == "Confirmed"  # Only confirmed blocks time.

            if same_booking == False and same_facility and same_date and same_time and active_booking:
                return True  # Time slot is taken.

        return False  # Time slot is available.

    def create_booking(self, facility_id, date, time_slot, number_of_people):
        if self.__current_user is None:
            raise BookingError("Please login first.")  # Must log in first.

        if self.__current_user.get_role() != "Student":
            raise BookingError("Only students can create bookings.")  # Only students book.

        if facility_id not in self.__facilities:
            raise BookingError("Facility was not found.")  # Facility must exist.

        facility = self.__facilities[facility_id]  # Gets selected facility.
        student = self.__current_user  # Gets current student.

        if facility.get_status() != "Available":
            raise BookingError("This facility is currently unavailable.")  # Blocks unavailable facility.

        self.check_access_permission(student, facility)  # Checks Standard/Premium rule.

        if date.strip() == "" or time_slot.strip() == "":
            raise ValidationError("Date and time cannot be empty.")  # Date/time required.

        if number_of_people <= 0:
            raise ValidationError("Number of people must be more than 0.")  # People must be positive.

        if number_of_people > facility.get_capacity():
            raise BookingError("Number of people cannot be more than the facility capacity.")  # Checks capacity.

        if self.is_time_slot_taken(facility_id, date, time_slot):
            raise BookingError("This facility is already booked at this date and time.")  # Stops duplicate booking.

        hours = self.calculate_hours_from_time_slot(time_slot)  # Calculates duration.
        total_cost = facility.calculate_cost(hours)  # Calculates cost.
        booking_id = "B" + str(len(self.__bookings) + 1)  # Creates booking ID.

        booking = Booking(booking_id, student.get_user_id(), facility_id, date, time_slot, hours, number_of_people, total_cost)  # Creates booking object.
        self.__bookings.append(booking)  # Adds booking to list.
        self.save_all_data()  # Saves booking.

        return booking  # Returns booking for receipt.

    def modify_booking(self, booking_id, date, time_slot, number_of_people):
        for booking in self.__bookings:  # Searches bookings.
            if booking.get_booking_id() == booking_id:  # Finds selected booking.
                facility = self.get_facility(booking.get_facility_id())  # Gets its facility.

                if booking.get_status() != "Confirmed":
                    raise BookingError("Cancelled bookings cannot be modified.")  # Cannot edit cancelled booking.

                if number_of_people <= 0:
                    raise ValidationError("Number of people must be more than 0.")  # People must be positive.

                if number_of_people > facility.get_capacity():
                    raise BookingError("Number of people cannot be more than the facility capacity.")  # Checks capacity.

                if self.is_time_slot_taken(booking.get_facility_id(), date, time_slot, booking_id):
                    raise BookingError("This facility is already booked at this date and time.")  # Checks conflict.

                hours = self.calculate_hours_from_time_slot(time_slot)  # Recalculates duration.
                total_cost = facility.calculate_cost(hours)  # Recalculates cost.
                booking.update_booking(date, time_slot, hours, number_of_people, total_cost)  # Updates booking.
                self.save_all_data()  # Saves changes.
                return booking  # Returns updated booking.

        raise BookingError("Booking was not found.")  # Booking ID not found.

    def get_my_bookings(self):
        my_bookings = []  # List for current student's bookings.

        for booking in self.__bookings:  # Checks all bookings.
            if booking.get_student_id() == self.__current_user.get_user_id():
                my_bookings.append(booking)  # Adds booking if it belongs to current student.

        return my_bookings  # Returns student's bookings.

    def cancel_booking(self, booking_id):
        for booking in self.__bookings:  # Searches bookings.
            if booking.get_booking_id() == booking_id:  # Finds selected booking.
                booking.cancel_booking()  # Changes status to Cancelled.
                self.save_all_data()  # Saves cancellation.
                return

        raise BookingError("Booking was not found.")  # Booking ID not found.

    def update_facility_status(self, facility_id, status):
        if facility_id not in self.__facilities:
            raise ValidationError("Facility was not found.")  # Facility must exist.

        self.__facilities[facility_id].set_status(status)  # Changes status.
        self.save_all_data()  # Saves status change.

    def booking_activity_per_day(self):
        activity = {}  # Stores date and booking count.

        for booking in self.__bookings:
            if booking.get_status() == "Confirmed":  # Counts confirmed bookings only.
                date = booking.get_date()  # Gets date.

                if date not in activity:
                    activity[date] = 0  # Starts count for new date.

                activity[date] = activity[date] + 1  # Adds one booking.

        return activity  # Returns report data.

    def facility_usage(self):
        usage = {}  # Stores facility ID and booking count.

        for facility_id in self.__facilities:
            usage[facility_id] = 0  # Starts every facility at 0.

        for booking in self.__bookings:
            if booking.get_status() == "Confirmed":  # Counts confirmed bookings only.
                facility_id = booking.get_facility_id()  # Gets facility ID.
                usage[facility_id] = usage[facility_id] + 1  # Adds one use.

        return usage  # Returns report data.

In [ ]:
# ---------------------------------------------------------
# Code Cell 4
# BookingGUI setup, login, register, student dashboard, profile, and facilities.
# This part builds the first half of the GUI.
# ---------------------------------------------------------

class BookingGUI:
    def __init__(self):
        self.system = BookingSystem()  # Creates the system logic object.
        self.root = tk.Tk()  # Creates main app window.
        self.root.title("Smart Campus Facility Booking System")  # Sets window title.
        self.root.geometry("1200x700")  # Sets window size.
        self.root.resizable(False, False)  # Stops resizing so layout stays stable.
        self.show_login_screen()  # Opens login screen first.

    def clear_screen(self):
        for widget in self.root.winfo_children():  # Goes through all widgets.
            widget.destroy()  # Removes old widgets.

    def show_title(self, title):
        tk.Label(self.root, text=title, font=("Arial", 20, "bold")).pack(pady=15)  # Shows screen title.

    def show_info(self, text):
        tk.Label(self.root, text=text, font=("Arial", 10), wraplength=900, justify="center", fg="gray30").pack(pady=5)  # Shows short instruction.

    def get_facility_id_from_choice(self, choice):
        return choice.split("] ")[1].split(" - ")[0]  # Extracts ID from text like [STANDARD] F1 - Study Room A.

    def show_login_screen(self):
        self.clear_screen()  # Clears screen.
        self.show_title("Smart Campus Facility Booking System")  # Shows title.
        self.show_info("Log in to book facilities or manage the campus booking system.")  # Shows instruction.

        frame = tk.Frame(self.root)  # Creates form frame.
        frame.pack(pady=20)  # Places frame.

        tk.Label(frame, text="User ID:").grid(row=0, column=0, pady=5, sticky="w")  # User ID label.
        user_id_entry = tk.Entry(frame, width=30)  # User ID entry.
        user_id_entry.grid(row=0, column=1, pady=5)  # Places entry.

        tk.Label(frame, text="Password:").grid(row=1, column=0, pady=5, sticky="w")  # Password label.
        password_entry = tk.Entry(frame, width=30, show="*")  # Password entry.
        password_entry.grid(row=1, column=1, pady=5)  # Places password entry.

        def login_action():
            try:
                user = self.system.login(user_id_entry.get(), password_entry.get())  # Attempts login.

                if user.get_role() == "Administrator":
                    self.show_admin_screen()  # Opens admin dashboard.
                else:
                    self.show_student_screen()  # Opens student dashboard.

            except LoginError as error:
                messagebox.showerror("Login Error", str(error))  # Shows login error.

        tk.Button(frame, text="Login", width=22, command=login_action).grid(row=2, column=0, columnspan=2, pady=10)  # Login button.
        tk.Button(frame, text="Create Account", width=22, command=self.show_register_screen).grid(row=3, column=0, columnspan=2)  # Register button.

    def show_register_screen(self):
        self.clear_screen()  # Clears screen.
        self.show_title("Create Account")  # Shows title.
        self.show_info("Create a student or administrator account for the booking system.")  # Shows instruction.

        frame = tk.Frame(self.root)  # Creates form frame.
        frame.pack(pady=20)  # Places frame.

        labels = ["User ID:", "Name:", "Email:", "Password:"]  # Form labels.
        entries = []  # Stores entry boxes.

        for i in range(len(labels)):
            tk.Label(frame, text=labels[i]).grid(row=i, column=0, pady=5, sticky="w")  # Creates label.
            entry = tk.Entry(frame, width=30)  # Creates entry.
            entry.grid(row=i, column=1, pady=5)  # Places entry.
            entries.append(entry)  # Stores entry.

        tk.Label(frame, text="User Type:").grid(row=4, column=0, pady=5, sticky="w")  # User type label.
        user_type_box = ttk.Combobox(frame, values=["Student", "Administrator"], state="readonly", width=27)  # User type dropdown.
        user_type_box.set("Student")  # Default user type.
        user_type_box.grid(row=4, column=1, pady=5)  # Places dropdown.

        tk.Label(frame, text="Access Type:").grid(row=5, column=0, pady=5, sticky="w")  # Access type label.
        access_box = ttk.Combobox(frame, values=["Standard", "Premium"], state="readonly", width=27)  # Access dropdown.
        access_box.set("Standard")  # Default access.
        access_box.grid(row=5, column=1, pady=5)  # Places access dropdown.

        def register_action():
            try:
                self.system.register_user(entries[0].get(), entries[1].get(), entries[2].get(), entries[3].get(), user_type_box.get(), access_box.get())  # Creates account.
                messagebox.showinfo("Success", "Account created successfully.")  # Shows success.
                self.show_login_screen()  # Returns to login.

            except ValidationError as error:
                messagebox.showerror("Register Error", str(error))  # Shows input error.

        tk.Button(frame, text="Create Account", width=20, command=register_action).grid(row=6, column=0, columnspan=2, pady=10)  # Create account button.
        tk.Button(frame, text="Back", width=20, command=self.show_login_screen).grid(row=7, column=0, columnspan=2)  # Back button.

    def show_student_screen(self):
        self.clear_screen()  # Clears screen.
        user = self.system.get_current_user()  # Gets logged-in user.
        self.show_title("Student Dashboard")  # Shows title.

        tk.Label(self.root, text="Welcome, " + user.get_name() + "!", font=("Arial", 14, "bold")).pack(pady=5)  # Welcome message.
        tk.Label(self.root, text="Role: " + user.get_role() + "   |   Access Type: " + user.get_access_type(), font=("Arial", 11)).pack(pady=2)  # Role/access.
        self.show_info("Choose what you want to do from the options below.")  # Shows instruction.

        frame = tk.Frame(self.root)  # Button frame.
        frame.pack(pady=25)  # Places frame.

        tk.Button(frame, text="View Facilities", width=30, command=self.show_facilities_screen).pack(pady=7)  # Opens facilities.
        tk.Button(frame, text="Book Facility", width=30, command=self.show_booking_screen).pack(pady=7)  # Opens booking screen.
        tk.Button(frame, text="My Bookings", width=30, command=self.show_my_bookings_screen).pack(pady=7)  # Opens my bookings.
        tk.Button(frame, text="View / Update My Details", width=30, command=self.show_profile_screen).pack(pady=7)  # Opens profile.
        tk.Button(frame, text="Logout", width=30, command=self.show_login_screen).pack(pady=18)  # Logs out.

    def show_profile_screen(self):
        self.clear_screen()  # Clears screen.
        self.show_title("My User Details")  # Shows title.
        self.show_info("View your account information and update your name or email.")  # Shows instruction.

        user = self.system.get_current_user()  # Gets current user.
        frame = tk.Frame(self.root)  # Creates form frame.
        frame.pack(pady=20)  # Places frame.

        tk.Label(frame, text="User ID:").grid(row=0, column=0, pady=5, sticky="w")  # User ID label.
        tk.Label(frame, text=user.get_user_id()).grid(row=0, column=1, pady=5, sticky="w")  # User ID value.
        tk.Label(frame, text="Role:").grid(row=1, column=0, pady=5, sticky="w")  # Role label.
        tk.Label(frame, text=user.get_role()).grid(row=1, column=1, pady=5, sticky="w")  # Role value.
        tk.Label(frame, text="Access Type:").grid(row=2, column=0, pady=5, sticky="w")  # Access label.
        tk.Label(frame, text=user.get_access_type()).grid(row=2, column=1, pady=5, sticky="w")  # Access value.

        tk.Label(frame, text="Name:").grid(row=3, column=0, pady=5, sticky="w")  # Name label.
        name_entry = tk.Entry(frame, width=30)  # Name entry.
        name_entry.insert(0, user.get_name())  # Fills current name.
        name_entry.grid(row=3, column=1, pady=5)  # Places entry.

        tk.Label(frame, text="Email:").grid(row=4, column=0, pady=5, sticky="w")  # Email label.
        email_entry = tk.Entry(frame, width=30)  # Email entry.
        email_entry.insert(0, user.get_email())  # Fills current email.
        email_entry.grid(row=4, column=1, pady=5)  # Places entry.

        def update_action():
            try:
                self.system.update_current_user_profile(name_entry.get(), email_entry.get())  # Saves profile.
                messagebox.showinfo("Updated", "Your details were updated.")  # Success message.
                self.show_student_screen()  # Back to dashboard.
            except ValidationError as error:
                messagebox.showerror("Error", str(error))  # Shows error.

        def upgrade_action():
            try:
                self.system.upgrade_current_user_access()  # Upgrades current student.
                messagebox.showinfo("Upgraded", "Your access type is now Premium.")  # Success message.
                self.show_profile_screen()  # Refreshes profile.
            except ValidationError as error:
                messagebox.showerror("Error", str(error))  # Shows error.

        tk.Button(frame, text="Save Changes", width=22, command=update_action).grid(row=5, column=0, columnspan=2, pady=5)  # Save button.

        if user.get_access_type() == "Premium":
            tk.Button(frame, text="Already Premium", width=22, state="disabled").grid(row=6, column=0, columnspan=2, pady=5)  # Disabled button.
        else:
            tk.Button(frame, text="Upgrade to Premium", width=22, command=upgrade_action).grid(row=6, column=0, columnspan=2, pady=5)  # Upgrade button.

        tk.Button(frame, text="Back", width=22, command=self.show_student_screen).grid(row=7, column=0, columnspan=2, pady=5)  # Back button.

    def show_facilities_screen(self):
        self.clear_screen()  # Clears screen.
        self.show_title("Facilities")  # Shows title.
        self.show_info("Facilities marked Premium require Premium access before booking.")  # Shows instruction.

        table_frame = tk.Frame(self.root)  # Creates table frame.
        table_frame.pack(pady=10)  # Places frame.

        columns = ("Access", "ID", "Name", "Type", "Location", "Capacity", "Price", "Status", "Time Slots")  # Table columns.
        tree = ttk.Treeview(table_frame, columns=columns, show="headings", height=12)  # Creates table.

        for column in columns:
            tree.heading(column, text=column)  # Sets column heading.

            if column == "Time Slots":
                tree.column(column, width=250)  # Wider time slots column.
            elif column == "Location":
                tree.column(column, width=140)  # Location width.
            elif column == "Name":
                tree.column(column, width=140)  # Name width.
            elif column == "Access":
                tree.column(column, width=100)  # Access width.
            elif column == "Price":
                tree.column(column, width=120)  # Price width.
            else:
                tree.column(column, width=90)  # Default width.

        for facility_id, facility in self.system.get_all_facilities().items():
            tree.insert("", "end", values=(facility.get_required_access(), facility_id, facility.get_name(), facility.get_facility_type(), facility.get_location(), facility.get_capacity(), str(facility.get_price_per_hour()) + " AED/hour", facility.get_status(), ", ".join(facility.get_time_slots())))  # Adds facility row.

        tree.pack()  # Places table.

        back_command = self.show_student_screen  # Default back for students.

        if self.system.get_current_user().get_role() == "Administrator":
            back_command = self.show_admin_screen  # Admin goes back to admin dashboard.

        tk.Button(self.root, text="Back", width=20, command=back_command).pack(pady=15)  # Back button.

In [ ]:
# ---------------------------------------------------------
# Code Cell 5
# Booking, My Bookings, and Modify Booking screens.
# This part lets students book, view, modify, and cancel bookings.
# ---------------------------------------------------------

    def show_booking_screen(self):
        self.clear_screen()  # Clears screen.
        self.show_title("Book a Facility")  # Shows title.
        self.show_info("Choose a facility and time slot. The system will calculate the duration and cost automatically.")  # Shows instruction.

        frame = tk.Frame(self.root)  # Creates form frame.
        frame.pack(pady=10)  # Places frame.

        tk.Label(frame, text="Facility:").grid(row=0, column=0, pady=5, sticky="w")  # Facility label.
        facility_choices = []  # Stores dropdown choices.

        for facility_id, facility in self.system.get_all_facilities().items():
            choice = "[" + facility.get_required_access().upper() + "] " + facility_id + " - " + facility.get_name()  # Builds dropdown text.
            facility_choices.append(choice)  # Adds choice.

        facility_box = ttk.Combobox(frame, values=facility_choices, state="readonly", width=38)  # Facility dropdown.
        facility_box.grid(row=0, column=1, pady=5)  # Places dropdown.

        if len(facility_choices) > 0:
            facility_box.set(facility_choices[0])  # Selects first facility.

        tk.Label(frame, text="Date:").grid(row=1, column=0, pady=5, sticky="w")  # Date label.
        date_choices = ["15 March 2026", "16 March 2026", "17 March 2026", "18 March 2026", "19 March 2026"]  # Date choices.
        date_box = ttk.Combobox(frame, values=date_choices, state="readonly", width=38)  # Date dropdown.
        date_box.grid(row=1, column=1, pady=5)  # Places dropdown.
        date_box.set(date_choices[0])  # Selects first date.

        tk.Label(frame, text="Time Slot:").grid(row=2, column=0, pady=5, sticky="w")  # Time slot label.
        time_box = ttk.Combobox(frame, state="readonly", width=38)  # Time slot dropdown.
        time_box.grid(row=2, column=1, pady=5)  # Places dropdown.

        duration_label = tk.Label(frame, text="Duration: 0 hour(s)", font=("Arial", 11, "bold"))  # Duration label.
        duration_label.grid(row=3, column=0, columnspan=2, pady=5)  # Places duration.
        cost_label = tk.Label(frame, text="Total Cost: 0 AED", font=("Arial", 11, "bold"))  # Cost label.
        cost_label.grid(row=4, column=0, columnspan=2, pady=5)  # Places cost.

        tk.Label(frame, text="Number of People:").grid(row=5, column=0, pady=5, sticky="w")  # People label.
        people_entry = tk.Entry(frame, width=41)  # People entry.
        people_entry.grid(row=5, column=1, pady=5)  # Places entry.

        def update_cost():
            try:
                facility_id = self.get_facility_id_from_choice(facility_box.get())  # Gets facility ID.
                facility = self.system.get_facility(facility_id)  # Gets facility object.
                time_slot = time_box.get()  # Gets selected time.
                hours = self.system.calculate_hours_from_time_slot(time_slot)  # Calculates duration.
                total_cost = facility.calculate_cost(hours)  # Calculates cost.
                duration_label.config(text="Duration: " + str(hours) + " hour(s)")  # Updates duration display.
                cost_label.config(text="Total Cost: " + str(total_cost) + " AED")  # Updates cost display.
            except:
                duration_label.config(text="Duration: 0 hour(s)")  # Safe default.
                cost_label.config(text="Total Cost: 0 AED")  # Safe default.

        def update_time_slots(event=None):
            facility_id = self.get_facility_id_from_choice(facility_box.get())  # Gets facility ID.
            facility = self.system.get_facility(facility_id)  # Gets facility object.
            time_box["values"] = facility.get_time_slots()  # Updates time dropdown.

            if len(facility.get_time_slots()) > 0:
                time_box.set(facility.get_time_slots()[0])  # Selects first time slot.

            update_cost()  # Updates duration and cost.

        facility_box.bind("<<ComboboxSelected>>", update_time_slots)  # Runs when facility changes.
        time_box.bind("<<ComboboxSelected>>", lambda event: update_cost())  # Runs when time changes.
        update_time_slots()  # Loads time slots on screen open.

        def book_action():
            try:
                facility_id = self.get_facility_id_from_choice(facility_box.get())  # Gets facility ID.
                number_of_people = int(people_entry.get())  # Converts people to number.
                booking = self.system.create_booking(facility_id, date_box.get(), time_box.get(), number_of_people)  # Creates booking.
                facility = self.system.get_facility(facility_id)  # Gets facility for receipt.
                user = self.system.get_current_user()  # Gets user for receipt.

                receipt = (
                    "SMART CAMPUS BOOKING RECEIPT\n"
                    "----------------------------------\n"
                    "Booking ID: " + booking.get_booking_id() + "\n"
                    "User Name: " + user.get_name() + "\n"
                    "User ID: " + user.get_user_id() + "\n"
                    "Access Type: " + user.get_access_type() + "\n\n"
                    "Facility: " + facility.get_name() + "\n"
                    "Required Access: " + facility.get_required_access() + "\n"
                    "Facility Type: " + facility.get_facility_type() + "\n"
                    "Location: " + facility.get_location() + "\n"
                    "Facility Capacity: " + str(facility.get_capacity()) + "\n\n"
                    "Date: " + booking.get_date() + "\n"
                    "Time Slot: " + booking.get_time_slot() + "\n"
                    "Duration: " + str(booking.get_hours()) + " hour(s)\n"
                    "Number of People: " + str(booking.get_number_of_people()) + "\n\n"
                    "Rate: " + str(facility.get_price_per_hour()) + " AED/hour\n"
                    "Total Cost: " + str(booking.get_total_cost()) + " AED\n"
                    "Payment Status: " + booking.get_payment_status() + "\n"
                    "Booking Status: " + booking.get_status()
                )  # Builds receipt text.

                messagebox.showinfo("Booking Receipt", receipt)  # Shows receipt.
                self.show_student_screen()  # Returns to dashboard.

            except ValueError:
                messagebox.showerror("Input Error", "Number of people must be a number.")  # Handles non-number input.
            except (BookingError, ValidationError) as error:
                messagebox.showerror("Booking Error", str(error))  # Handles booking/input errors.

        tk.Button(frame, text="Confirm Booking", width=22, command=book_action).grid(row=6, column=0, columnspan=2, pady=10)  # Confirm button.
        tk.Button(frame, text="Back", width=22, command=self.show_student_screen).grid(row=7, column=0, columnspan=2, pady=5)  # Back button.

    def show_my_bookings_screen(self):
        self.clear_screen()  # Clears screen.
        self.show_title("My Bookings")  # Shows title.
        self.show_info("Select a booking if you want to modify or cancel it.")  # Shows instruction.

        table_frame = tk.Frame(self.root)  # Table frame.
        table_frame.pack(pady=10)  # Places frame.

        columns = ("Booking ID", "Facility ID", "Date", "Time", "Duration", "People", "Cost", "Status")  # Table columns.
        tree = ttk.Treeview(table_frame, columns=columns, show="headings", height=12)  # Creates table.

        for column in columns:
            tree.heading(column, text=column)  # Sets heading.
            tree.column(column, width=130)  # Sets width.

        for booking in self.system.get_my_bookings():
            tree.insert("", "end", values=(booking.get_booking_id(), booking.get_facility_id(), booking.get_date(), booking.get_time_slot(), str(booking.get_hours()) + " hour(s)", booking.get_number_of_people(), str(booking.get_total_cost()) + " AED", booking.get_status()))  # Adds booking row.

        tree.pack()  # Places table.

        def get_selected_booking_id():
            selected = tree.focus()  # Gets selected row.

            if selected == "":
                messagebox.showerror("Error", "Please select a booking first.")  # No selection error.
                return None

            return str(tree.item(selected)["values"][0])  # Returns booking ID.

        def cancel_selected():
            booking_id = get_selected_booking_id()  # Gets selected booking.

            if booking_id is None:
                return  # Stops if no booking selected.

            try:
                self.system.cancel_booking(booking_id)  # Cancels booking.
                messagebox.showinfo("Cancelled", "Booking cancelled successfully.")  # Success message.
                self.show_my_bookings_screen()  # Refreshes screen.
            except BookingError as error:
                messagebox.showerror("Error", str(error))  # Shows error.

        def modify_selected():
            booking_id = get_selected_booking_id()  # Gets selected booking.

            if booking_id is None:
                return  # Stops if no booking selected.

            self.show_modify_booking_screen(booking_id)  # Opens modify screen.

        tk.Button(self.root, text="Modify Selected Booking", width=25, command=modify_selected).pack(pady=5)  # Modify button.
        tk.Button(self.root, text="Cancel Selected Booking", width=25, command=cancel_selected).pack(pady=5)  # Cancel button.
        tk.Button(self.root, text="Back", width=20, command=self.show_student_screen).pack(pady=5)  # Back button.

    def show_modify_booking_screen(self, booking_id):
        selected_booking = None  # Starts with no booking selected.

        for booking in self.system.get_my_bookings():
            if booking.get_booking_id() == booking_id:
                selected_booking = booking  # Finds selected booking.

        if selected_booking is None:
            messagebox.showerror("Error", "Booking was not found.")  # Shows error if not found.
            return

        self.clear_screen()  # Clears screen.
        self.show_title("Modify Booking")  # Shows title.
        self.show_info("Change the date, time slot, or number of people for this booking.")  # Shows instruction.

        facility = self.system.get_facility(selected_booking.get_facility_id())  # Gets facility.
        frame = tk.Frame(self.root)  # Creates form frame.
        frame.pack(pady=10)  # Places frame.

        tk.Label(frame, text="Booking ID:").grid(row=0, column=0, pady=5, sticky="w")  # Booking ID label.
        tk.Label(frame, text=selected_booking.get_booking_id()).grid(row=0, column=1, pady=5, sticky="w")  # Booking ID value.
        tk.Label(frame, text="Facility:").grid(row=1, column=0, pady=5, sticky="w")  # Facility label.
        tk.Label(frame, text=facility.get_name()).grid(row=1, column=1, pady=5, sticky="w")  # Facility value.

        tk.Label(frame, text="Date:").grid(row=2, column=0, pady=5, sticky="w")  # Date label.
        date_choices = ["15 March 2026", "16 March 2026", "17 March 2026", "18 March 2026", "19 March 2026"]  # Date choices.
        date_box = ttk.Combobox(frame, values=date_choices, state="readonly", width=35)  # Date dropdown.
        date_box.grid(row=2, column=1, pady=5)  # Places dropdown.
        date_box.set(selected_booking.get_date())  # Sets current date.

        tk.Label(frame, text="Time Slot:").grid(row=3, column=0, pady=5, sticky="w")  # Time label.
        time_box = ttk.Combobox(frame, values=facility.get_time_slots(), state="readonly", width=35)  # Time dropdown.
        time_box.grid(row=3, column=1, pady=5)  # Places dropdown.
        time_box.set(selected_booking.get_time_slot())  # Sets current time.

        duration_label = tk.Label(frame, text="", font=("Arial", 11, "bold"))  # Duration label.
        duration_label.grid(row=4, column=0, columnspan=2, pady=5)  # Places duration.
        cost_label = tk.Label(frame, text="", font=("Arial", 11, "bold"))  # Cost label.
        cost_label.grid(row=5, column=0, columnspan=2, pady=5)  # Places cost.

        tk.Label(frame, text="Number of People:").grid(row=6, column=0, pady=5, sticky="w")  # People label.
        people_entry = tk.Entry(frame, width=38)  # People entry.
        people_entry.insert(0, str(selected_booking.get_number_of_people()))  # Fills current people.
        people_entry.grid(row=6, column=1, pady=5)  # Places entry.

        def update_modify_cost(event=None):
            time_slot = time_box.get()  # Gets time slot.
            hours = self.system.calculate_hours_from_time_slot(time_slot)  # Calculates duration.
            total_cost = facility.calculate_cost(hours)  # Calculates cost.
            duration_label.config(text="Duration: " + str(hours) + " hour(s)")  # Updates duration.
            cost_label.config(text="Total Cost: " + str(total_cost) + " AED")  # Updates cost.

        time_box.bind("<<ComboboxSelected>>", update_modify_cost)  # Updates when time changes.
        update_modify_cost()  # Shows current cost/duration.

        def save_changes():
            try:
                number_of_people = int(people_entry.get())  # Converts people to number.
                self.system.modify_booking(selected_booking.get_booking_id(), date_box.get(), time_box.get(), number_of_people)  # Saves changes.
                messagebox.showinfo("Updated", "Booking updated successfully.")  # Success message.
                self.show_my_bookings_screen()  # Back to bookings.
            except ValueError:
                messagebox.showerror("Input Error", "Number of people must be a number.")  # Handles non-number.
            except (BookingError, ValidationError) as error:
                messagebox.showerror("Error", str(error))  # Shows error.

        tk.Button(frame, text="Save Changes", width=22, command=save_changes).grid(row=7, column=0, columnspan=2, pady=10)  # Save button.
        tk.Button(frame, text="Back", width=22, command=self.show_my_bookings_screen).grid(row=8, column=0, columnspan=2)  # Back button.

In [ ]:
# ---------------------------------------------------------
# Code Cell 6
# Admin screens and run program.
# This part builds admin dashboard, all bookings, user management, status update, reports, and starts the app.
# ---------------------------------------------------------

    def show_admin_screen(self):
        self.clear_screen()  # Clears screen.
        self.show_title("Admin Dashboard")  # Shows title.
        current_user = self.system.get_current_user()  # Gets admin user.
        tk.Label(self.root, text="Welcome, " + current_user.get_name() + "!", font=("Arial", 14, "bold")).pack(pady=5)  # Welcome.
        self.show_info("Choose an admin action below.")  # Shows instruction.

        frame = tk.Frame(self.root)  # Button frame.
        frame.pack(pady=25)  # Places frame.

        tk.Button(frame, text="View Facilities", width=30, command=self.show_facilities_screen).pack(pady=7)  # Facilities.
        tk.Button(frame, text="View All Bookings", width=30, command=self.show_all_bookings_screen).pack(pady=7)  # All bookings.
        tk.Button(frame, text="Manage Users", width=30, command=self.show_manage_users_screen).pack(pady=7)  # Manage users.
        tk.Button(frame, text="Update Facility Status", width=30, command=self.show_update_status_screen).pack(pady=7)  # Update status.
        tk.Button(frame, text="Reports: Activity and Usage", width=30, command=self.show_reports_screen).pack(pady=7)  # Reports.
        tk.Button(frame, text="Logout", width=30, command=self.show_login_screen).pack(pady=18)  # Logout.

    def show_all_bookings_screen(self):
        self.clear_screen()  # Clears screen.
        self.show_title("All Bookings")  # Shows title.
        self.show_info("This screen shows all booking records in the system.")  # Shows instruction.

        table_frame = tk.Frame(self.root)  # Table frame.
        table_frame.pack(pady=10)  # Places frame.

        columns = ("Booking ID", "Student ID", "Facility ID", "Date", "Time", "Duration", "People", "Cost", "Status")  # Columns.
        tree = ttk.Treeview(table_frame, columns=columns, show="headings", height=12)  # Creates table.

        for column in columns:
            tree.heading(column, text=column)  # Sets heading.
            tree.column(column, width=120)  # Sets width.

        for booking in self.system.get_all_bookings():
            tree.insert("", "end", values=(booking.get_booking_id(), booking.get_student_id(), booking.get_facility_id(), booking.get_date(), booking.get_time_slot(), str(booking.get_hours()) + " hour(s)", booking.get_number_of_people(), str(booking.get_total_cost()) + " AED", booking.get_status()))  # Adds booking row.

        tree.pack()  # Places table.
        tk.Button(self.root, text="Back", width=20, command=self.show_admin_screen).pack(pady=15)  # Back button.

    def show_manage_users_screen(self):
        self.clear_screen()  # Clears screen.
        self.show_title("Manage Users")  # Shows title.
        self.show_info("Select a user to upgrade a student account or remove a user.")  # Shows instruction.

        table_frame = tk.Frame(self.root)  # Table frame.
        table_frame.pack(pady=10)  # Places frame.

        columns = ("User ID", "Name", "Email", "Role", "Access Type")  # User columns.
        tree = ttk.Treeview(table_frame, columns=columns, show="headings", height=12)  # Creates table.

        for column in columns:
            tree.heading(column, text=column)  # Sets heading.
            tree.column(column, width=210)  # Sets width.

        for user_id, user in self.system.get_all_users().items():
            tree.insert("", "end", values=(user_id, user.get_name(), user.get_email(), user.get_role(), user.get_access_type()))  # Adds user row.

        tree.pack()  # Places table.

        def get_selected_user_id():
            selected = tree.focus()  # Gets selected row.

            if selected == "":
                messagebox.showerror("Error", "Please select a user first.")  # No selection error.
                return None

            return str(tree.item(selected)["values"][0])  # Returns user ID.

        def upgrade_selected():
            user_id = get_selected_user_id()  # Gets selected user.

            if user_id is None:
                return  # Stops if none selected.

            try:
                self.system.upgrade_access(user_id)  # Upgrades student.
                messagebox.showinfo("Upgraded", "Student access upgraded to Premium.")  # Success.
                self.show_manage_users_screen()  # Refreshes table.
            except ValidationError as error:
                messagebox.showerror("Error", str(error))  # Shows error.

        def delete_selected():
            user_id = get_selected_user_id()  # Gets selected user.

            if user_id is None:
                return  # Stops if none selected.

            try:
                self.system.delete_user(user_id)  # Deletes user.
                messagebox.showinfo("Deleted", "User deleted successfully.")  # Success.
                self.show_manage_users_screen()  # Refreshes table.
            except ValidationError as error:
                messagebox.showerror("Error", str(error))  # Shows error.

        tk.Button(self.root, text="Upgrade Selected Student", width=25, command=upgrade_selected).pack(pady=5)  # Upgrade button.
        tk.Button(self.root, text="Delete Selected User", width=25, command=delete_selected).pack(pady=5)  # Delete button.
        tk.Button(self.root, text="Back", width=20, command=self.show_admin_screen).pack(pady=5)  # Back button.

    def show_update_status_screen(self):
        self.clear_screen()  # Clears screen.
        self.show_title("Update Facility Status")  # Shows title.
        self.show_info("Change whether a facility is available or unavailable for booking.")  # Shows instruction.

        frame = tk.Frame(self.root)  # Form frame.
        frame.pack(pady=20)  # Places frame.

        facility_choices = []  # Facility dropdown list.

        for facility_id, facility in self.system.get_all_facilities().items():
            facility_choices.append(facility_id + " - " + facility.get_name())  # Adds facility choice.

        tk.Label(frame, text="Facility:").grid(row=0, column=0, pady=5, sticky="w")  # Facility label.
        facility_box = ttk.Combobox(frame, values=facility_choices, state="readonly", width=35)  # Facility dropdown.
        facility_box.grid(row=0, column=1, pady=5)  # Places dropdown.

        if len(facility_choices) > 0:
            facility_box.set(facility_choices[0])  # Selects first facility.

        tk.Label(frame, text="Status:").grid(row=1, column=0, pady=5, sticky="w")  # Status label.
        status_box = ttk.Combobox(frame, values=["Available", "Unavailable"], state="readonly", width=35)  # Status dropdown.
        status_box.set("Available")  # Default status.
        status_box.grid(row=1, column=1, pady=5)  # Places dropdown.

        def update_action():
            try:
                facility_id = facility_box.get().split(" - ")[0]  # Gets facility ID.
                self.system.update_facility_status(facility_id, status_box.get())  # Updates status.
                messagebox.showinfo("Updated", "Facility status updated successfully.")  # Success.
                self.show_admin_screen()  # Back to admin dashboard.
            except ValidationError as error:
                messagebox.showerror("Error", str(error))  # Shows error.

        tk.Button(frame, text="Update", width=22, command=update_action).grid(row=2, column=0, columnspan=2, pady=10)  # Update button.
        tk.Button(frame, text="Back", width=22, command=self.show_admin_screen).grid(row=3, column=0, columnspan=2)  # Back button.

    def show_reports_screen(self):
        self.clear_screen()  # Clears screen.
        self.show_title("Reports")  # Shows title.
        self.show_info("This report summarizes booking activity and facility usage.")  # Shows instruction.

        text_box = tk.Text(self.root, width=120, height=25)  # Creates report text box.
        text_box.pack(pady=10)  # Places text box.

        text_box.insert(tk.END, "BOOKING ACTIVITY PER DAY\n")  # Report section title.
        text_box.insert(tk.END, "------------------------\n")  # Divider.

        activity = self.system.booking_activity_per_day()  # Gets activity report.

        if len(activity) == 0:
            text_box.insert(tk.END, "No confirmed bookings yet.\n")  # No bookings message.
        else:
            for date in activity:
                text_box.insert(tk.END, date + ": " + str(activity[date]) + " booking(s)\n")  # Adds daily booking count.

        text_box.insert(tk.END, "\nFACILITY USAGE AND CAPACITY\n")  # Second report section.
        text_box.insert(tk.END, "---------------------------\n")  # Divider.

        usage = self.system.facility_usage()  # Gets usage report.

        for facility_id, count in usage.items():
            facility = self.system.get_facility(facility_id)  # Gets facility object.
            line = facility.get_name() + " | Required Access: " + facility.get_required_access() + " | Type: " + facility.get_facility_type() + " | Capacity: " + str(facility.get_capacity()) + " | Confirmed Bookings: " + str(count) + " | Status: " + facility.get_status() + "\n"  # Builds report line.
            text_box.insert(tk.END, line)  # Adds report line.

        text_box.config(state="disabled")  # Makes report read-only.
        tk.Button(self.root, text="Back", width=20, command=self.show_admin_screen).pack(pady=10)  # Back button.

    def run(self):
        self.root.mainloop()  # Keeps Tkinter window open.


# Creates the app object.
app = BookingGUI()

# Starts the application.
app.run()